In [2]:
# ============================================================
# CELL 0 — Imports and setup
# ============================================================
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
from IPython.display import display
import os
import sys
sys.path.append('..')

# Publication style
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

CITIES = ['manhattan', 'pittsburgh', 'philadelphia']
CITY_COLORS = {'manhattan': '#2196F3', 'pittsburgh': '#FF9800', 'philadelphia': '#4CAF50'}
LABEL_COLORS = {'Answerable': '#4CAF50', 'Ambiguous': '#FF9800', 'Contradictory': '#F44336'}
VARIANT_COLORS = {'mask_landmark': '#9C27B0', 'mask_directions': '#2196F3', 'mask_both': '#FF5722'}

BASE_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__) 
           if '__file__' in dir() else os.getcwd(), '..'))
REPORTS_DIR = os.path.join(BASE_DIR, 'reports', 'llm_audits')
DATA_DIR    = os.path.join(BASE_DIR, 'data')

INPUT_PATH  = os.path.join(REPORTS_DIR, 'LLM_DEGRADATION_INPUT.parquet')
RESULTS_PATH = os.path.join(REPORTS_DIR, 'LLM_DEGRADATION_RESULTS.parquet')

os.makedirs(f'{REPORTS_DIR}/figures', exist_ok=True)

# ── Staleness check ──────────────────────────────────────────
for label, path in [('INPUT', INPUT_PATH), ('RESULTS', RESULTS_PATH)]:
    if os.path.exists(path):
        mtime = os.path.getmtime(path)
        ts    = pd.Timestamp(mtime, unit='s', tz='UTC').strftime('%Y-%m-%d %H:%M:%S UTC')
        size  = os.path.getsize(path) / 1024
        print(f"📄 {label}: {path}")
        print(f"   modified={ts}  size={size:.1f} KB")
    else:
        print(f"❌ {label} NOT FOUND: {path}")

print("\n✅ Setup complete")

📄 INPUT: /vol/joberant_nobck/data/NLP_368307701_2526a/adanassi/nlp-allocentric-spatial-reasoning/reports/llm_audits/LLM_DEGRADATION_INPUT.parquet
   modified=2026-04-30 18:30:45 UTC  size=2336.2 KB
📄 RESULTS: /vol/joberant_nobck/data/NLP_368307701_2526a/adanassi/nlp-allocentric-spatial-reasoning/reports/llm_audits/LLM_DEGRADATION_RESULTS.parquet
   modified=2026-05-01 11:08:47 UTC  size=1568.9 KB

✅ Setup complete


In [3]:
import pandas as pd

df = pd.read_parquet('../reports/llm_audits/LLM_DEGRADATION_RESULTS_LARGE.parquet')

print("Output distribution:")
print(df['llm_output_raw'].value_counts().head(20))

print("\nMask token outputs:")
mask_out = df['llm_output_raw'].str.contains(
    r'\[MASK\]|\[DIR_MASK\]', na=False, regex=True)
print(f"  Contains mask token: {mask_out.sum()}/50 ({mask_out.mean():.1%})")

print("\nResolution breakdown:")
print(df.groupby('resolution_succeeded')['llm_output_raw'].count())

Output distribution:
llm_output_raw
[DIR_MASK]                            13
[MASK]                                13
[MASK] on Liberty Street               2
The Josephine Shaw Lowell Fountain     1
76th Street Basketball Courts          1
[MASK] on [DIR_MASK] 40th Street       1
[DIR_MASK] 40th Street                 1
[MASK] on West 40th Street             1
[DIR_MASK] 51st Street                 1
Baxter Street                          1
Chambers Street                        1
Chambers [MASK]                        1
Forsyth Street                         1
[MASK] on 2nd Avenue                   1
The [DIR_MASK] Catholic church         1
Liberty Street                         1
Alexander and Bonin gallery            1
[MASK] on York Avenue                  1
York Avenue                            1
[MASK] on York Avenue.                 1
Name: count, dtype: int64

Mask token outputs:
  Contains mask token: 41/50 (82.0%)

Resolution breakdown:
resolution_succeeded
False    47
True

In [4]:
print(df.groupby('variant_type')['llm_output_raw'].apply(
    lambda x: x.str.contains(r'\[MASK\]|\[DIR_MASK\]', 
    regex=True, na=False).mean()).round(3))

variant_type
mask_both          1.000
mask_directions    0.500
mask_landmark      0.944
Name: llm_output_raw, dtype: float64


In [6]:
import pandas as pd
df = pd.read_parquet('../reports/llm_audits/LLM_DEGRADATION_RESULTS_LARGE.parquet')

mask_out = df['llm_output_raw'].str.contains(
    r'\[MASK\]|\[DIR_MASK\]', regex=True, na=False)
print(f"Still containing mask tokens: {mask_out.sum()}/50 ({mask_out.mean():.1%})")
print("\nTop outputs:")
print(df['llm_output_raw'].value_counts().head(15))

Still containing mask tokens: 11/50 (22.0%)

Top outputs:
llm_output_raw
[MASK]                              8
Liberty Street                      3
76th Street Basketball Courts       3
1st Avenue                          3
Lotte Hotels & Resorts              3
Gato restaurant                     3
Josephine Shaw Lowell [MASK]        2
Creperie                            2
10th Avenue                         2
Asian restaurant                    2
Chambers                            2
[MASK] on [DIR_MASK] 40th Street    1
Blank Fitness                       1
Chambers Street                     1
Forsyth Street                      1
Name: count, dtype: int64
